# 01 — Lectura y Discovery

**Objetivo del notebook:** primer contacto con los datos. Cargar ambos datasets (etiquetado y sin etiquetar), entender sus dimensiones, tipos de datos, y hacer un primer inventario de qué tenemos antes de cualquier análisis o limpieza.

Este notebook no modifica nada — solo observa y documenta el estado inicial.

## Contexto del proyecto

El objetivo del proyecto es predecir si una persona **fuma** (`smoking` = 1) o **no fuma** (`smoking` = 0) a partir de variables biométricas y de exámenes médicos.

Tenemos dos archivos:
- `smoking_prediction.xlsx` — datos **etiquetados** (con la columna `smoking`). Sirve para entrenar y validar.
- `smoking_prediction_entrega.xlsx` — datos **sin etiquetar** (sin `smoking`). Sobre estos generaremos la predicción final.

La métrica de evaluación es el **F1-Score para la clase target = 1** (fumadores).

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print('Librerías cargadas ✓')

Librerías cargadas ✓


## 1. Carga de los datos

Cargamos ambos archivos desde `data/raw/`, que contiene los datos originales tal como se descargaron, sin ninguna modificación.

In [2]:
df_train = pd.read_excel('../data/raw/smoking_prediction.xlsx')
df_entrega = pd.read_excel('../data/raw/smoking_prediction_entrega.xlsx')

print(f'Dataset etiquetado (train):    {df_train.shape[0]:,} filas × {df_train.shape[1]} columnas')
print(f'Dataset sin etiquetar (entrega): {df_entrega.shape[0]:,} filas × {df_entrega.shape[1]} columnas')

Dataset etiquetado (train):    50,000 filas × 27 columnas
Dataset sin etiquetar (entrega): 5,692 filas × 26 columnas


## 2. Primer vistazo al dataset etiquetado

In [3]:
df_train.head()

,ID,gender,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,relaxation,fasting blood sugar,Cholesterol,triglyceride,HDL,LDL,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,oral,dental caries,tartar,smoking
0,0,F,40,155,60,3.377083,0.043056,0.041667,0.041667,0.041667,4.750000,3.041667,3.916667,8.958333,3.416667,3.041667,5.250000,0.506250,0.041667,0.004861,0.750000,0.791667,1.125000,Y,0,Y,0
1,1,F,40,160,60,3.375000,0.005556,0.004167,0.041667,0.041667,4.958333,2.916667,5.416667,8.000000,4.791667,1.750000,5.291667,0.504861,0.041667,0.004167,0.916667,0.791667,0.750000,Y,0,Y,0
2,2,M,55,170,60,3.333333,0.005556,0.005556,0.041667,0.041667,5.750000,3.583333,3.708333,10.083333,7.583333,2.291667,6.291667,0.630556,0.041667,0.041667,0.875000,0.666667,0.916667,Y,0,N,1
3,3,M,40,165,70,3.666667,0.045139,0.045139,0.041667,0.041667,4.166667,2.500000,4.000000,13.416667,10.583333,1.875000,9.416667,0.588194,0.041667,0.041667,0.791667,1.083333,0.750000,Y,0,Y,0
4,4,F,40,155,60,3.583333,0.041667,0.041667,0.041667,0.041667,5.000000,3.083333,3.333333,7.666667,3.083333,2.583333,4.458333,0.503472,0.041667,0.004167,0.666667,0.583333,0.916667,Y,0,N,0


In [4]:
# Tipos de datos y memoria
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 27 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   50000 non-null  int64  
 1   gender               50000 non-null  str    
 2   age                  50000 non-null  int64  
 3   height(cm)           50000 non-null  int64  
 4   weight(kg)           50000 non-null  int64  
 5   waist(cm)            50000 non-null  float64
 6   eyesight(left)       50000 non-null  float64
 7   eyesight(right)      50000 non-null  float64
 8   hearing(left)        50000 non-null  float64
 9   hearing(right)       50000 non-null  float64
 10  systolic             50000 non-null  float64
 11  relaxation           50000 non-null  float64
 12  fasting blood sugar  50000 non-null  float64
 13  Cholesterol          50000 non-null  float64
 14  triglyceride         50000 non-null  float64
 15  HDL                  50000 non-null  float64
 1

## 3. Inventario de columnas

Listamos todas las columnas y su tipo para entender qué información tenemos disponible.

In [5]:
# Descripción de cada columna del dataset
descripcion_columnas = {
    'ID': 'Identificador único de la persona',
    'gender': 'Sexo biológico (M/F)',
    'age': 'Edad',
    'height(cm)': 'Altura en centímetros',
    'weight(kg)': 'Peso en kilogramos',
    'waist(cm)': 'Circunferencia de cintura',
    'eyesight(left)': 'Agudeza visual ojo izquierdo',
    'eyesight(right)': 'Agudeza visual ojo derecho',
    'hearing(left)': 'Audición oído izquierdo',
    'hearing(right)': 'Audición oído derecho',
    'systolic': 'Presión arterial sistólica',
    'relaxation': 'Presión arterial diastólica',
    'fasting blood sugar': 'Glucosa en ayunas',
    'Cholesterol': 'Colesterol total',
    'triglyceride': 'Triglicéridos',
    'HDL': 'Colesterol HDL (bueno)',
    'LDL': 'Colesterol LDL (malo)',
    'hemoglobin': 'Hemoglobina en sangre',
    'Urine protein': 'Proteína en orina',
    'serum creatinine': 'Creatinina en suero',
    'AST': 'Enzima hepática AST',
    'ALT': 'Enzima hepática ALT',
    'Gtp': 'Enzima hepática GTP',
    'oral': 'Examen oral realizado (Y/N)',
    'dental caries': 'Presencia de caries dental (0/1)',
    'tartar': 'Presencia de sarro (Y/N)',
    'smoking': 'TARGET — fuma (1) o no fuma (0)'
}

for col in df_train.columns:
    desc = descripcion_columnas.get(col, 'sin descripción')
    print(f'  {col:<22} {str(df_train[col].dtype):<10} {desc}')

  ID                     int64      Identificador único de la persona
  gender                 str        Sexo biológico (M/F)
  age                    int64      Edad
  height(cm)             int64      Altura en centímetros
  weight(kg)             int64      Peso en kilogramos
  waist(cm)              float64    Circunferencia de cintura
  eyesight(left)         float64    Agudeza visual ojo izquierdo
  eyesight(right)        float64    Agudeza visual ojo derecho
  hearing(left)          float64    Audición oído izquierdo
  hearing(right)         float64    Audición oído derecho
  systolic               float64    Presión arterial sistólica
  relaxation             float64    Presión arterial diastólica
  fasting blood sugar    float64    Glucosa en ayunas
  Cholesterol            float64    Colesterol total
  triglyceride           float64    Triglicéridos
  HDL                    float64    Colesterol HDL (bueno)
  LDL                    float64    Colesterol LDL (malo)
  hemoglob

## 4. Distribución del target

Verificamos cómo está distribuida la variable que queremos predecir. Esto es crítico: si las clases están muy desbalanceadas, hay que tenerlo en cuenta al entrenar y al elegir la métrica.

In [6]:
conteo = df_train['smoking'].value_counts().sort_index()
porcentaje = df_train['smoking'].value_counts(normalize=True).sort_index() * 100

print('Distribución del target (smoking):')
print(f'  No fumadores (0): {conteo[0]:,} ({porcentaje[0]:.1f}%)')
print(f'  Fumadores (1):    {conteo[1]:,} ({porcentaje[1]:.1f}%)')
print()
print(f'Ratio de desbalance: {conteo[0]/conteo[1]:.2f} : 1')

Distribución del target (smoking):
  No fumadores (0): 31,671 (63.3%)
  Fumadores (1):    18,329 (36.7%)

Ratio de desbalance: 1.73 : 1


El target está **desbalanceado**: hay casi el doble de no fumadores que de fumadores. Esto tiene dos consecuencias importantes:

1. Al hacer el split train/test usaremos `stratify` para mantener esta proporción en ambos conjuntos.
2. La métrica F1 sobre la clase 1 es la adecuada justamente porque el accuracy sería engañoso — un modelo que prediga siempre "no fuma" tendría ~63% de accuracy sin aprender nada.

## 5. Verificación de integridad básica

In [7]:
print('── Dataset etiquetado ──')
print(f'  Valores nulos totales: {df_train.isnull().sum().sum()}')
print(f'  Filas duplicadas:      {df_train.duplicated().sum()}')
print()
print('── Dataset sin etiquetar ──')
print(f'  Valores nulos totales: {df_entrega.isnull().sum().sum()}')
print(f'  Filas duplicadas:      {df_entrega.duplicated().sum()}')

── Dataset etiquetado ──
  Valores nulos totales: 0
  Filas duplicadas:      0

── Dataset sin etiquetar ──
  Valores nulos totales: 0
  Filas duplicadas:      0


## 6. Comparación de columnas entre ambos datasets

Es fundamental que el dataset de entrega tenga las mismas columnas que el de entrenamiento (salvo el target). Si hubiera diferencias, el pipeline de preprocesamiento fallaría al aplicarse sobre los datos nuevos.

In [8]:
cols_train = set(df_train.columns) - {'smoking'}
cols_entrega = set(df_entrega.columns)

print(f'Columnas en train (sin target): {len(cols_train)}')
print(f'Columnas en entrega:            {len(cols_entrega)}')
print()
print(f'¿Coinciden exactamente?: {cols_train == cols_entrega}')
print()

solo_train = cols_train - cols_entrega
solo_entrega = cols_entrega - cols_train

if solo_train:
    print(f'Solo en train: {solo_train}')
if solo_entrega:
    print(f'Solo en entrega: {solo_entrega}')
if not solo_train and not solo_entrega:
    print('✓ Ambos datasets tienen exactamente las mismas features.')

Columnas en train (sin target): 26
Columnas en entrega:            26

¿Coinciden exactamente?: True

✓ Ambos datasets tienen exactamente las mismas features.


## Conclusiones del discovery

- El dataset etiquetado tiene **50.000 filas y 27 columnas** (26 features + 1 target).
- El dataset de entrega tiene **5.692 filas y 26 columnas** (las mismas features, sin target).
- **No hay valores nulos ni duplicados** en ninguno de los dos datasets.
- El target está **desbalanceado** (~63% no fumadores, ~37% fumadores).
- Las columnas coinciden exactamente entre ambos datasets, lo que garantiza que el pipeline será aplicable.

**Observaciones para los siguientes notebooks:**
- La columna `ID` es un identificador y deberá eliminarse antes de modelar.
- La columna `oral` debe revisarse en el EDA (sospecha de que tiene un solo valor).
- Las columnas `gender`, `oral` y `tartar` son categóricas y requerirán encoding.
- Hay que investigar en el EDA qué variables aportan más señal al target.